# 03 — Train KT Models (Ablation Study)

8 ablation configs:
| # | Tên | Model | Label | Params |
|---|-----|-------|-------|--------|
| 0 | Baseline | Majority | binary | — |
| 1 | BKT | Bayesian | binary | 4/KC, L-BFGS-B |
| 2 | DKT-Binary | LSTM | binary (BCE) | h=128, L=1, d=0.2 |
| 3 | DKT-Quality★ | LSTM | quality (MSE) | h=128, L=1, d=0.2 |
| 4 | SAKT-Binary | Attention | binary (BCE) | e=64, H=4, L=1, d=0.2 |
| 5 | SAKT-Quality★ | Attention | quality (MSE) | e=64, H=4, L=1, d=0.2 |
| 6 | DKT-Deep | LSTM deep | binary (BCE) | h=256, L=2, d=0.3 |
| 7 | SAKT-Deep | Attention deep | binary (BCE) | e=128, H=4, L=2, d=0.2 |

Metrics: ROC-AUC, PR-AUC, RMSE, Accuracy  
Split: train=700 / val=150 / test=150 users

In [ ]:
import sys, json, pickle, time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, mean_squared_error, average_precision_score

ON_KAGGLE = Path('/kaggle').exists()

if ON_KAGGLE:
    KG_DATASET = Path('/kaggle/input/datasets/zakhim/rcmsys/rcmsys_dataset')
    sys.path.insert(0, str(KG_DATASET))
    PROC     = Path('/kaggle/input/notebooks/zakhim/02-feature-engineering-ipynb/proj/data/processed')
    RESULTS  = Path('/kaggle/working/results')
else:
    ROOT     = Path().resolve().parent
    sys.path.insert(0, str(ROOT))
    PROC     = ROOT / 'data' / 'processed'
    RESULTS  = ROOT / 'results'

MODELS   = RESULTS / 'models'
METRICS  = RESULTS / 'metrics'
ABLATION = RESULTS / 'ablation'
PLOTS    = RESULTS / 'plots'

for p in [MODELS, METRICS, ABLATION, PLOTS]:
    p.mkdir(parents=True, exist_ok=True)

from kt_models.bkt  import BKT
from kt_models.dkt  import DKT
from kt_models.sakt import SAKT

print('ON_KAGGLE:', ON_KAGGLE)
print('PROC    :', PROC)
print('MODELS  :', MODELS)
print('ABLATION:', ABLATION)


In [ ]:
with open(PROC / 'metadata.json') as f:
    meta = json.load(f)

with open(PROC / 'train_seqs.pkl', 'rb') as f:
    train_seqs = pickle.load(f)
with open(PROC / 'val_seqs.pkl', 'rb') as f:
    val_seqs = pickle.load(f)
with open(PROC / 'test_seqs.pkl', 'rb') as f:
    test_seqs = pickle.load(f)
with open(PROC / 'coldstart.pkl', 'rb') as f:
    coldstart = pickle.load(f)

N_KCS = meta['n_kcs']
print(f'N_KCS={N_KCS}, n_users={meta["n_users"]}, n_questions={meta["n_questions"]}')
print(f'train={len(train_seqs)}, val={len(val_seqs)}, test={len(test_seqs)}')

In [ ]:
# ── Data Integrity Checks ───────────────────────────────────────────────────
from collections import Counter

PASS = '✅'; WARN = '⚠️ '

# 1. Leakage: user overlap
tr_ids = set(s['user_id'] for s in train_seqs)
va_ids = set(s['user_id'] for s in val_seqs)
te_ids = set(s['user_id'] for s in test_seqs)
leak = len(tr_ids & va_ids) + len(tr_ids & te_ids) + len(va_ids & te_ids)
print(f'{PASS if leak==0 else WARN} [1] User leakage: train∩val={len(tr_ids&va_ids)}, train∩test={len(tr_ids&te_ids)}, val∩test={len(va_ids&te_ids)}')

# 2. Label distribution — cả 2 class trong val/test
for split_name, seqs in [('val', val_seqs), ('test', test_seqs)]:
    labels = [int(c >= 1) for s in seqs for c in s['correct_seq']]
    uniq = set(labels)
    ok = len(uniq) == 2
    pos_rate = np.mean(labels)
    print(f'{PASS if ok else WARN} [2] {split_name} label dist: classes={sorted(uniq)}, pos_rate={pos_rate:.3f}')

# 3. Sequence length distribution
for split_name, seqs in [('train', train_seqs), ('val', val_seqs), ('test', test_seqs)]:
    lens = [s['seq_len'] for s in seqs]
    min_len = min(lens)
    ok = min_len >= 2
    print(f'{PASS if ok else WARN} [3] {split_name} seq_len: min={min_len}, max={max(lens)}, mean={np.mean(lens):.1f}')

# 4. KC frequency drift
def kc_dist(seqs):
    kcs = [kc for s in seqs for kc in s['kc_seq']]
    total = len(kcs)
    return {k: 100*v/total for k, v in Counter(kcs).items()}

tr_d, va_d, te_d = kc_dist(train_seqs), kc_dist(val_seqs), kc_dist(test_seqs)
drifts = {kc: max(abs(tr_d.get(kc,0)-va_d.get(kc,0)), abs(tr_d.get(kc,0)-te_d.get(kc,0)))
          for kc in range(N_KCS)}
flagged = {kc: round(d,1) for kc, d in drifts.items() if d > 3}
ok = len(flagged) == 0
print(f'{PASS if ok else WARN} [4] KC drift >3%: {len(flagged)} KCs — {flagged}')

# 5. Class imbalance warning
all_train_labels = [int(c >= 1) for s in train_seqs for c in s['correct_seq']]
pos_rate = np.mean(all_train_labels)
imbalanced = pos_rate > 0.75
print(f'{WARN if imbalanced else PASS} [5] Class imbalance: pos_rate={pos_rate:.3f}',
      '→ report PR-AUC alongside ROC-AUC' if imbalanced else '')

# 6. Quality distribution consistency
for split_name, seqs in [('train', train_seqs), ('val', val_seqs), ('test', test_seqs)]:
    qs = [q for s in seqs for q in s['quality_seq']]
    dist = {v: round(100*qs.count(v)/len(qs),1) for v in [0.0, 0.5, 1.0]}
    print(f'{PASS} [6] {split_name} quality dist: {dist}')

# 7. Coldstart coverage
all_uids = set(s['user_id'] for s in train_seqs + val_seqs + test_seqs)
missing_cs = all_uids - set(coldstart.keys())
print(f'{PASS if len(missing_cs)==0 else WARN} [7] Coldstart: {len(coldstart)}/{len(all_uids)} users, missing={len(missing_cs)}')

print('\nIntegrity checks complete.')

In [ ]:
def collect_binary_labels(seqs):
    y = []
    for seq in seqs:
        y.extend([int(c >= 1) for c in seq['correct_seq']])
    return np.array(y)

def compute_prauc(y_true, y_pred):
    """PR-AUC (Average Precision). Robust khi class imbalance cao."""
    try:
        if len(np.unique(y_true)) < 2: return 0.0
        return round(float(average_precision_score(y_true, y_pred)), 4)
    except:
        return 0.0

def add_prauc(model, seqs, metrics_dict):
    """Tính PR-AUC và thêm vào metrics dict inplace."""
    yt, yp = model.predict(seqs)
    yb = yt if model.mode == 'binary' else (yt >= 0.5).astype(int)
    metrics_dict['prauc'] = compute_prauc(yb, yp)
    return metrics_dict

all_results = []

def record(name, val_m, test_m, elapsed):
    row = {'model': name, 'elapsed_s': round(elapsed, 1)}
    row.update({f'val_{k}': v for k, v in val_m.items()})
    row.update({f'test_{k}': v for k, v in test_m.items()})
    all_results.append(row)
    print(f'  [val]  {val_m}')
    print(f'  [test] {test_m}')
    print(f'  time:  {elapsed:.1f}s\n')

In [ ]:
# === Config 0: Baseline (majority class) ===
print('=== Baseline ===')
t0 = time.time()

y_train = collect_binary_labels(train_seqs)
majority_p = float(y_train.mean())

def baseline_eval(seqs, p):
    y = collect_binary_labels(seqs)
    y_pred = np.full_like(y, p, dtype=float)
    auc   = round(roc_auc_score(y, y_pred), 4) if len(np.unique(y)) > 1 else 0.5
    prauc = compute_prauc(y, y_pred)
    rmse  = round(float(np.sqrt(mean_squared_error(y, y_pred))), 4)
    acc   = round(float(np.mean((y_pred >= 0.5).astype(int) == y)), 4)
    return {'auc': auc, 'prauc': prauc, 'rmse': rmse, 'acc': acc}

val_m  = baseline_eval(val_seqs, majority_p)
test_m = baseline_eval(test_seqs, majority_p)
record('Baseline', val_m, test_m, time.time()-t0)

In [ ]:
# === Config 1: BKT ===
print('=== BKT ===')
t0 = time.time()

bkt = BKT(n_kcs=N_KCS, seed=42)
bkt.fit(train_seqs, verbose=True)
bkt.save(str(MODELS / 'bkt.npy'))

val_m  = bkt.evaluate(val_seqs)
test_m = bkt.evaluate(test_seqs)
yt, yp = bkt.predict(val_seqs)
val_m['prauc']  = compute_prauc(yt, yp)
yt, yp = bkt.predict(test_seqs)
test_m['prauc'] = compute_prauc(yt, yp)
record('BKT', val_m, test_m, time.time()-t0)

with open(METRICS / 'bkt.json', 'w') as f:
    json.dump({'val': val_m, 'test': test_m}, f, indent=2)

In [ ]:
# === Config 2: DKT-Binary (h=128, L=1) ===
print('=== DKT-Binary ===')
t0 = time.time()

dkt_bin = DKT(n_kcs=N_KCS, mode='binary',
              hidden_size=128, n_layers=1, dropout=0.2,
              lr=1e-3, batch_size=64, epochs=30, patience=5, seed=42)
dkt_bin.fit(train_seqs, val_seqs, verbose=True)
dkt_bin.save(str(MODELS / 'dkt_binary.pt'))

val_m  = add_prauc(dkt_bin, val_seqs,  dkt_bin.evaluate(val_seqs))
test_m = add_prauc(dkt_bin, test_seqs, dkt_bin.evaluate(test_seqs))
record('DKT-Binary', val_m, test_m, time.time()-t0)

with open(METRICS / 'dkt_binary.json', 'w') as f:
    json.dump({'val': val_m, 'test': test_m, 'history': dkt_bin.history}, f, indent=2)

In [ ]:
# === Config 3: DKT-Quality★ (h=128, L=1, MSE) ===
print('=== DKT-Quality★ ===')
t0 = time.time()

dkt_q = DKT(n_kcs=N_KCS, mode='quality',
            hidden_size=128, n_layers=1, dropout=0.2,
            lr=1e-3, batch_size=64, epochs=30, patience=5, seed=42)
dkt_q.fit(train_seqs, val_seqs, verbose=True)
dkt_q.save(str(MODELS / 'dkt_quality.pt'))

val_m  = add_prauc(dkt_q, val_seqs,  dkt_q.evaluate(val_seqs))
test_m = add_prauc(dkt_q, test_seqs, dkt_q.evaluate(test_seqs))
record('DKT-Quality★', val_m, test_m, time.time()-t0)

with open(METRICS / 'dkt_quality.json', 'w') as f:
    json.dump({'val': val_m, 'test': test_m, 'history': dkt_q.history}, f, indent=2)

In [ ]:
# === Config 4: SAKT-Binary (e=64, H=4, L=1) ===
print('=== SAKT-Binary ===')
t0 = time.time()

sakt_bin = SAKT(n_kcs=N_KCS, mode='binary',
                embed_dim=64, n_heads=4, n_layers=1, dropout=0.2,
                lr=1e-3, batch_size=64, epochs=30, patience=5, seed=42)
sakt_bin.fit(train_seqs, val_seqs, verbose=True)
sakt_bin.save(str(MODELS / 'sakt_binary.pt'))

val_m  = add_prauc(sakt_bin, val_seqs,  sakt_bin.evaluate(val_seqs))
test_m = add_prauc(sakt_bin, test_seqs, sakt_bin.evaluate(test_seqs))
record('SAKT-Binary', val_m, test_m, time.time()-t0)

with open(METRICS / 'sakt_binary.json', 'w') as f:
    json.dump({'val': val_m, 'test': test_m, 'history': sakt_bin.history}, f, indent=2)

In [ ]:
# === Config 5: SAKT-Quality★ (e=64, H=4, L=1, MSE) ===
print('=== SAKT-Quality★ ===')
t0 = time.time()

sakt_q = SAKT(n_kcs=N_KCS, mode='quality',
              embed_dim=64, n_heads=4, n_layers=1, dropout=0.2,
              lr=1e-3, batch_size=64, epochs=30, patience=5, seed=42)
sakt_q.fit(train_seqs, val_seqs, verbose=True)
sakt_q.save(str(MODELS / 'sakt_quality.pt'))

val_m  = add_prauc(sakt_q, val_seqs,  sakt_q.evaluate(val_seqs))
test_m = add_prauc(sakt_q, test_seqs, sakt_q.evaluate(test_seqs))
record('SAKT-Quality★', val_m, test_m, time.time()-t0)

with open(METRICS / 'sakt_quality.json', 'w') as f:
    json.dump({'val': val_m, 'test': test_m, 'history': sakt_q.history}, f, indent=2)

In [ ]:
# === Config 6: DKT-Deep (h=256, L=2, dropout=0.3) ===
print('=== DKT-Deep ===')
t0 = time.time()

dkt_deep = DKT(n_kcs=N_KCS, mode='binary',
               hidden_size=256, n_layers=2, dropout=0.3,
               lr=5e-4, batch_size=64, epochs=30, patience=5, seed=42)
dkt_deep.fit(train_seqs, val_seqs, verbose=True)
dkt_deep.save(str(MODELS / 'dkt_deep.pt'))

val_m  = add_prauc(dkt_deep, val_seqs,  dkt_deep.evaluate(val_seqs))
test_m = add_prauc(dkt_deep, test_seqs, dkt_deep.evaluate(test_seqs))
record('DKT-Deep', val_m, test_m, time.time()-t0)

with open(METRICS / 'dkt_deep.json', 'w') as f:
    json.dump({'val': val_m, 'test': test_m, 'history': dkt_deep.history}, f, indent=2)

In [ ]:
# === Config 7: SAKT-Deep (e=128, H=4, L=2) ===
print('=== SAKT-Deep ===')
t0 = time.time()

sakt_deep = SAKT(n_kcs=N_KCS, mode='binary',
                 embed_dim=128, n_heads=4, n_layers=2, dropout=0.2,
                 lr=5e-4, batch_size=64, epochs=30, patience=5, seed=42)
sakt_deep.fit(train_seqs, val_seqs, verbose=True)
sakt_deep.save(str(MODELS / 'sakt_deep.pt'))

val_m  = add_prauc(sakt_deep, val_seqs,  sakt_deep.evaluate(val_seqs))
test_m = add_prauc(sakt_deep, test_seqs, sakt_deep.evaluate(test_seqs))
record('SAKT-Deep', val_m, test_m, time.time()-t0)

with open(METRICS / 'sakt_deep.json', 'w') as f:
    json.dump({'val': val_m, 'test': test_m, 'history': sakt_deep.history}, f, indent=2)

In [ ]:
# === Ablation Summary ===
df = pd.DataFrame(all_results)
cols = ['model', 'val_auc', 'val_prauc', 'val_rmse', 'val_acc',
        'test_auc', 'test_prauc', 'test_rmse', 'test_acc', 'elapsed_s']
cols = [c for c in cols if c in df.columns]
df = df[cols]

print('\n=== ABLATION RESULTS ===')
print(df.to_string(index=False))

df.to_csv(ABLATION / 'ablation_results.csv', index=False)
df.to_json(ABLATION / 'ablation_results.json', orient='records', indent=2)
print('\nSaved → results/ablation/')

best_auc = df.loc[df['test_auc'].idxmax()]
print(f'\n★ Best test ROC-AUC: {best_auc["model"]} — {best_auc["test_auc"]:.4f}')
if 'test_prauc' in df.columns:
    best_pr = df.loc[df['test_prauc'].idxmax()]
    print(f'★ Best test PR-AUC:  {best_pr["model"]} — {best_pr["test_prauc"]:.4f}')

In [ ]:
# Learning curves (6 neural models)
import matplotlib.pyplot as plt

models_hist = [
    ('DKT-Binary',    dkt_bin.history),
    ('DKT-Quality★',  dkt_q.history),
    ('DKT-Deep',      dkt_deep.history),
    ('SAKT-Binary',   sakt_bin.history),
    ('SAKT-Quality★', sakt_q.history),
    ('SAKT-Deep',     sakt_deep.history),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, hist) in zip(axes.flat, models_hist):
    epochs = range(1, len(hist['train_loss']) + 1)
    ax2 = ax.twinx()
    ax.plot(epochs, hist['train_loss'], 'b-',  label='train loss', linewidth=1.5)
    ax.plot(epochs, hist['val_loss'],   'b--', label='val loss',   linewidth=1.5)
    ax2.plot(epochs, hist['val_auc'],   'r-',  label='val AUC',    linewidth=1.5)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss', color='b')
    ax2.set_ylabel('AUC', color='r')
    ax.tick_params(axis='y', labelcolor='b')
    ax2.tick_params(axis='y', labelcolor='r')
    ax.legend(loc='upper left', fontsize=7)
    ax2.legend(loc='upper right', fontsize=7)

plt.suptitle('Learning Curves — KT Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS / 'learning_curves.png', dpi=150)
plt.show()
print('Saved → results/plots/learning_curves.png')

In [ ]:
# AUC bar chart — so sánh tất cả 8 configs
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = plt.cm.tab10(np.linspace(0, 0.8, len(df)))

for ax, metric, title in [
    (axes[0], 'test_auc',   'Test ROC-AUC'),
    (axes[1], 'test_prauc', 'Test PR-AUC'),
]:
    if metric not in df.columns: continue
    bars = ax.barh(df['model'], df[metric], color=colors, edgecolor='white')
    ax.set_xlabel(metric.split('_',1)[1].upper())
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(0, 1.05)
    for bar, val in zip(bars, df[metric]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(PLOTS / 'ablation_bar.png', dpi=150)
plt.show()
print('Saved → results/plots/ablation_bar.png')